# Long-Term Memory (LTM) in LangGraph

**Long-term memory** allows a LangGraph application to persist information **across different conversations/threads**.

### Key concepts

* **Store** → LangGraph's mechanism for persistent memory.
* **Namespace** → Groups related memories, e.g. `("users", "123")`.
* **Key** → Unique identifier for a memory inside a namespace.
* **Value** → Actual information being stored.
* **Semantic search** → Finds relevant memories based on meaning rather than exact keywords.
* **Embeddings** → Converts text/memories into vectors so semantic similarity can be calculated.

### Basic flow

```text
User
 ↓
LangGraph
 ↓
Store
 ↓
Save important information
 ↓
Embedding + Vector Search
 ↓
Retrieve relevant memories
 ↓
LLM uses memories
```

### Example

```python
namespace = ("users", "123")

store.put(
    namespace,
    "1",
    {"data": "User prefers Python"}
)
```

Later:

```python
store.search(
    namespace,
    query="Which programming language does the user prefer?"
)
```

The search can retrieve **"User prefers Python"** even though the query doesn't contain the exact same wording.

**Mental model:**

> **Checkpointer = short-term/thread memory**
> **Store = long-term memory**
> **Namespace = where memories belong**
> **Semantic search = find relevant memories**


In [1]:
from langgraph.store.memory import BaseStore, InMemoryStore

# Create Memories 

In [3]:
# create store
store =  InMemoryStore()

### Namespace

In LangGraph Store, a namespace is a logical grouping/address for memories.

Think of it like a folder path in which your memories are stored.



#### Code Block
```python
namespace = ("users", "123", ...)  

store.put(
    namespace=namespace, # 
    key="1", 
    value={"data": "User prefers Python"}
)
```

```markdown
Database analogy

namespace  ≈ partition / collection
key        ≈ unique ID
value      ≈ actual document
```

> Namespace tells LangGraph WHERE to look; key tells it WHICH exact memory; value is WHAT is stored.

In [4]:
# create namespace :
# It should be in the form of Tuple

# namespace: namespace is not but the route or location details for data 
# or consider it as folder path
# example: ("user-1","preferencers") -> /user-1/preferencers/
# namespace is a way to orginize memory inside memory store

namespace_1 = ("user","u1") # alway tuple

In [5]:
# put method user to store data in a perticular namespace 
store.put(namespace=namespace_1, key="1", value={"data": "user likes pizza"})
store.put(namespace=namespace_1, key="2", value={"data": "user prefers dark mode"})


In [6]:
namespace_2 = ("user","u2") 
 
store.put(namespace=namespace_2, key="1", value={"data": "user likes pasta"})
store.put(namespace=namespace_2, key="2", value={"data": "user prefers grid style"})


# retrieving memory

In [7]:
# fetch an specific point(key) of memory 
store.get(namespace=namespace_2,key='2')

Item(namespace=['user', 'u2'], key='2', value={'data': 'user prefers grid style'}, created_at='2026-08-22T13:26:49.268425+00:00', updated_at='2026-08-22T13:26:49.268427+00:00')

# retrieve all memory

In [8]:
items = store.search(namespace_1)

for item in items:
    print(item)

Item(namespace=['user', 'u1'], key='1', value={'data': 'user likes pizza'}, created_at='2026-08-22T13:26:49.101805+00:00', updated_at='2026-08-22T13:26:49.101810+00:00', score=None)
Item(namespace=['user', 'u1'], key='2', value={'data': 'user prefers dark mode'}, created_at='2026-08-22T13:26:49.101983+00:00', updated_at='2026-08-22T13:26:49.101985+00:00', score=None)


# sementic search

Because long-term memory can contain a lot of memories, and the agent usually doesn't know the exact key or wording of the memory it needs.

Semantic search lets the agent retrieve memories based on meaning, not exact words.

> retrive only most relevent memory matches

- **Without samentic search**: Send all the memories with LLM. which produce hellucination and context overflow

- **with Sementic Search:** Send only relevent Memories

In [9]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embedding_model = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
)

In [10]:
store_with_embed = InMemoryStore(index={"embed": embedding_model, "dims": 768})

In [11]:
namespace_3 = ("users", "u3")

In [14]:
store_with_embed.put(
    namespace=namespace_3,
    key="1",
    value={"data": "User prefers Python for backend development."}
)

store_with_embed.put(
    namespace=namespace_3,
    key="2",
    value={"data": "User works with AWS cloud services."}
)

store_with_embed.put(
    namespace=namespace_3,
    key="3",
    value={"data": "User is learning LangGraph and LangChain."}
)

store_with_embed.put(
    namespace=namespace_3,
    key="4",
    value={"data": "User likes building event-driven systems."}
)

store_with_embed.put(
    namespace=namespace_3,
    key="5",
    value={"data": "User prefers simple and scalable architectures."}
)

store_with_embed.put(
    namespace=namespace_3,
    key="6",
    value={"data": "User has experience with DynamoDB and S3."}
)

store_with_embed.put(
    namespace=namespace_3,
    key="7",
    value={"data": "User is preparing for system design interviews."}
)

store_with_embed.put(
    namespace=namespace_3,
    key="8",
    value={"data": "User enjoys solving data structures problems."}
)

store_with_embed.put(
    namespace=namespace_3,
    key="8",
    value={"data": "User planning to go to mumbai."}
)

In [16]:
store_with_embed.search(
    namespace_3,
    query="i want to visit delhi",
    limit=3
)

[Item(namespace=['users', 'u3'], key='8', value={'data': 'User planning to go to mumbai.'}, created_at='2026-08-22T13:27:45.520256+00:00', updated_at='2026-08-22T13:27:45.520261+00:00', score=0.672790973758859),
 Item(namespace=['users', 'u3'], key='7', value={'data': 'User is preparing for system design interviews.'}, created_at='2026-08-22T13:27:43.715837+00:00', updated_at='2026-08-22T13:27:43.715840+00:00', score=0.6062816974125633),
 Item(namespace=['users', 'u3'], key='5', value={'data': 'User prefers simple and scalable architectures.'}, created_at='2026-08-22T13:27:42.267620+00:00', updated_at='2026-08-22T13:27:42.267623+00:00', score=0.5932909733829235)]